# 03 · Mapa interactivo de incidencias + catastro (Folium / Leaflet)

Renderiza un HTML con las incidencias deduplicadas sobre la red de alcantarillado.
Los puntos se colorean y dimensionan por **nº de reclamos** (la deduplicación hecha
visible); los multi-reclamo llevan el **número** y los críticos **pulsan**.

Toda la lógica del mapa vive en `src/epsel/mapa.py`; aquí solo la orquestamos.
**Entrada:** `incidencias_catastro.parquet` (del nb02). **Salida:**
`reports/mapa_incidencias.html`.

In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd

from epsel import config, mapa

## 1 · Cargar las incidencias con catastro (salida del nb02)

In [2]:
gdf = pd.read_parquet(config.PROCESSED / "incidencias_catastro.parquet")
print("Incidencias a mapear:", len(gdf))
print("\nDistribución por nº de reclamos:")
print(gdf["N_RECLAMOS"].value_counts().sort_index())

Incidencias a mapear: 10531

Distribución por nº de reclamos:
N_RECLAMOS
1    9999
2     489
3      38
4       3
5       1
6       1
Name: count, dtype: int64


## 2 · Reglas de presentación (definidas en `epsel.mapa`)

- **1 reclamo** → verde, punto simple (95% de los casos).
- **2 reclamos** → ámbar, con el número.
- **3+ reclamos** → rojo, con número **y animación de pulso**.

In [3]:
print("Umbral número:", mapa.UMBRAL_NUMERO, "| umbral pulso:", mapa.UMBRAL_PULSO)
n_num = (gdf["N_RECLAMOS"] >= mapa.UMBRAL_NUMERO).sum()
n_pulse = (gdf["N_RECLAMOS"] >= mapa.UMBRAL_PULSO).sum()
print(f"Con número: {n_num:,} | con pulso (críticos): {n_pulse:,}")

Umbral número: 2 | umbral pulso: 3
Con número: 532 | con pulso (críticos): 43


## 3 · Construir el mapa

`mapa.build_map` arma todo: capas de catastro, incidencias (capa GeoJSON liviana
para los simples + marcadores con contador para los multi), mapa de calor y leyenda.

In [ ]:
m = mapa.build_map(gdf)
m

## 4 · Guardar el HTML

In [5]:
out = config.REPORTS / "mapa_incidencias.html"
config.REPORTS.mkdir(exist_ok=True)
m.save(str(out))
print(f"Guardado {out.name}: {out.stat().st_size/1e6:.1f} MB")

Guardado mapa_incidencias.html: 6.1 MB
